# Video Frame Search

This notebook adds a cooking video to ApertureDB, extracts frames at regular intervals, embeds each frame with a CLIP model, and runs text-to-frame search to find the most relevant moments in the video.

## Connect to ApertureDB

**Option A: ApertureDB Cloud (recommended)**  
Sign up for a [free 30-day trial](https://cloud.aperturedata.io). Get your key from **Connect → Generate API Key**, add it to a `.env` file in this directory:
```
APERTUREDB_KEY=your_key_here
```

**Option B: Community Edition (local Docker)**  
Run this in a terminal before starting the notebook:
```bash
docker run -d --name aperturedb \
  -p 55555:55555 -e ADB_MASTER_KEY=admin -e ADB_FORCE_SSL=false \
  aperturedata/aperturedb-community
```

See [client configuration options](https://docs.aperturedata.io/Setup/client/configuration) for all connection methods and [server setup options](https://docs.aperturedata.io/category/setup-server) for deployment choices.

In [ ]:
%pip install aperturedb sentence-transformers Pillow python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# !adb config create localdb --active \
#     --host localhost --port 55555 \
#     --username admin --password admin \
#     --no-use-ssl --no-interactive

In [ ]:
from aperturedb.CommonLibrary import create_connector

client = create_connector()
response, _ = client.query([{"GetStatus": {}}])
client.print_last_response()

### Step: Load the CLIP model

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("clip-ViT-B-32")
print(f"Embedding dimensions: {model.get_sentence_embedding_dimension()}")

### Step: Add a video to ApertureDB

In [ ]:
VIDEO_URL = "https://github.com/aperture-data/Cookbook/raw/main/notebooks/simple/data/crepe_flambe.mp4"

q = [{
    "AddVideo": {
        "url": VIDEO_URL,
        "_ref": 1,
        "properties": {
            "name":    "crepe_flambe",
            "cuisine": "French",
        }
    }
}]
response, _ = client.query(q)
client.print_last_response()

### Step: Extract frames from the video

In [ ]:
# Extract frames at positions 0, 30, 60, 90, 120 (frame numbers)
FRAME_NUMBERS = [0, 30, 60, 90, 120]

q = [{
    "FindVideo": {
        "constraints": {"name": ["==", "crepe_flambe"]},
        "_ref": 1,
        "blobs": False,
    }
}, {
    "FindFrames": {
        "video_ref": 1,
        "frames":    FRAME_NUMBERS,
        "blobs":     True,
    }
}]

response, frame_blobs = client.query(q)
n_frames = response[1]["FindFrames"].get("returned", 0)
print(f"Extracted {n_frames} frames")

### Step: Create a DescriptorSet for frame embeddings

In [ ]:
SET_NAME = "video_frame_search"

client.query([{
    "AddDescriptorSet": {
        "name":       SET_NAME,
        "dimensions": 512,
        "engine":     "FaissFlat",
        "metric":     "CS",
    }
}])
client.print_last_response()

### Step: Embed and store each frame

Each frame blob is decoded as a PIL image, embedded with CLIP, and stored as a Descriptor linked to the video with the frame number as a property.

In [ ]:
from PIL import Image
from io import BytesIO
import numpy as np

for i, (frame_blob, frame_num) in enumerate(zip(frame_blobs, FRAME_NUMBERS)):
    img = Image.open(BytesIO(frame_blob)).convert("RGB")
    emb = model.encode(img, normalize_embeddings=True).astype("float32")

    q = [{
        "FindVideo": {
            "constraints": {"name": ["==", "crepe_flambe"]},
            "_ref": 1,
            "blobs": False,
        }
    }, {
        "AddDescriptor": {
            "set":   SET_NAME,
            "connect": {"ref": 1, "class": "has_frame_embedding"},
            "properties": {
                "frame_number": frame_num,
                "video_name":   "crepe_flambe",
            }
        }
    }]
    client.query(q, [emb.tobytes()])

print(f"Stored embeddings for {len(frame_blobs)} frames")

### Step: Text-to-frame search

Encode a text query and find the frame whose visual content best matches. The frame number tells you where in the video the match occurs.

In [ ]:
query_text = "pouring batter into pan"

query_emb = model.encode(query_text, normalize_embeddings=True).astype("float32")

q = [{
    "FindDescriptor": {
        "set":         SET_NAME,
        "k_neighbors": 3,
        "distances":   True,
        "results":     {"all_properties": True},
    }
}]

response, _ = client.query(q, [query_emb.tobytes()])

print(f'Query: "{query_text}"\n')
for entity in response[0]["FindDescriptor"].get("entities", []):
    print(f"  Frame {entity['frame_number']:>4}   distance={entity['_distance']:.4f}")

### Step: Cleanup (optional)

In [ ]:
client.query([{"DeleteDescriptorSet": {"with_name": SET_NAME}}])
client.print_last_response()